# Feature Engineering (Issue #6)

Builds pregame features on top of `data/full_dataset_clean.csv` and saves `data/features.csv`.

**Rule:** every feature must be knowable *before kickoff*. Columns describing the game's outcome
(points, postgame Elo, postgame win probability, excitement index, line scores) are dropped from the
model features to avoid leakage.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/full_dataset_clean.csv")
df["startDate"] = pd.to_datetime(df["startDate"], utc=True)
print(df.shape)

(2021, 51)


## 1. Matchup strength

In [2]:
POWER_CONFS = {"SEC", "Big Ten", "Big 12", "ACC", "Pac-12"}

df["elo_diff"] = df["homePregameElo"] - df["awayPregameElo"]
df["abs_elo_diff"] = df["elo_diff"].abs()          # closer matchups may draw more
df["combined_elo"] = df["homePregameElo"] + df["awayPregameElo"]
df["away_is_fcs"] = (df["awayClassification"] == "fcs").astype(int)
df["away_is_power"] = df["awayConference"].isin(POWER_CONFS).astype(int)
df["home_is_power"] = df["homeConference"].isin(POWER_CONFS).astype(int)
df[["elo_diff", "abs_elo_diff", "combined_elo", "away_is_fcs", "away_is_power"]].describe().T

,count,mean,std,min,25%,50%,75%,max
elo_diff,2021.0,26.790857,276.134003,-948.0,-150.0,23.0,196.0,1057.0
abs_elo_diff,2021.0,215.384551,174.798805,0.0,78.0,173.0,309.0,1057.0
combined_elo,2021.0,3002.176239,353.150701,1855.0,2772.0,2988.0,3212.0,4360.0
away_is_fcs,2021.0,0.173182,0.378498,0.0,0.0,0.0,0.0,1.0
away_is_power,2021.0,0.385453,0.486823,0.0,0.0,0.0,1.0,1.0


## 2. Rivalry game flag

There is no rivalry column in the data, so this uses a hand-built list of well-known rivalries.
Pairs are unordered. Add any your team thinks are missing.

In [3]:
RIVALRIES = [
    ("Alabama", "Auburn"), ("Michigan", "Ohio State"), ("Georgia", "Florida"),
    ("Georgia", "Georgia Tech"), ("Florida", "Florida State"), ("Clemson", "South Carolina"),
    ("Texas", "Texas A&M"), ("Texas", "Oklahoma"), ("Oklahoma", "Oklahoma State"),
    ("USC", "UCLA"), ("USC", "Notre Dame"), ("Notre Dame", "Navy"), ("Army", "Navy"),
    ("Oregon", "Oregon State"), ("Washington", "Washington State"), ("Washington", "Oregon"),
    ("California", "Stanford"), ("Arizona", "Arizona State"), ("Utah", "BYU"),
    ("Ole Miss", "Mississippi State"), ("LSU", "Alabama"), ("LSU", "Texas A&M"),
    ("Tennessee", "Alabama"), ("Tennessee", "Vanderbilt"), ("Tennessee", "Kentucky"),
    ("Kentucky", "Louisville"), ("Iowa", "Iowa State"), ("Iowa", "Minnesota"),
    ("Iowa", "Nebraska"), ("Minnesota", "Wisconsin"), ("Indiana", "Purdue"),
    ("Michigan", "Michigan State"), ("Illinois", "Northwestern"), ("Penn State", "Pittsburgh"),
    ("Pittsburgh", "West Virginia"), ("Virginia", "Virginia Tech"), ("North Carolina", "NC State"),
    ("North Carolina", "Duke"), ("Miami", "Florida State"), ("Kansas", "Kansas State"),
    ("Kansas", "Missouri"), ("Baylor", "TCU"), ("Texas Tech", "TCU"), ("SMU", "TCU"),
    ("Colorado", "Colorado State"), ("Boise State", "Fresno State"), ("Air Force", "Army"),
    ("Air Force", "Navy"), ("Cincinnati", "Louisville"), ("Rutgers", "Maryland"),
    ("Arkansas", "LSU"), ("Auburn", "Georgia"), ("Missouri", "Arkansas"),
    ("Toledo", "Bowling Green"), ("Central Michigan", "Western Michigan"),
    ("Utah", "Utah State"), ("New Mexico", "New Mexico State"), ("Memphis", "Ole Miss"),
]
RIVAL_SET = {frozenset(p) for p in RIVALRIES}

df["rivalry_game"] = [
    int(frozenset((h, a)) in RIVAL_SET) for h, a in zip(df["homeTeam"], df["awayTeam"])
]
print("rivalry games:", df["rivalry_game"].sum())
df.loc[df["rivalry_game"] == 1, ["season", "homeTeam", "awayTeam"]].head(10)

rivalry games: 107


,season,homeTeam,awayTeam
129,2024,Iowa,Iowa State
195,2024,Oregon State,Oregon
198,2024,Washington,Washington State
199,2024,Pittsburgh,West Virginia
208,2024,Utah State,Utah
230,2024,Colorado State,Colorado
279,2024,SMU,TCU
294,2024,Minnesota,Iowa
336,2024,Duke,North Carolina
353,2024,New Mexico State,New Mexico


## 3. Form entering the game: win streak, last result, season win %

Each team plays both home and away games, so we stack both sides into one long table,
sort each team's season by date, and use `.shift(1)` so a game's features only use
results from *earlier* games (no peeking at the current result).

In [4]:
home_side = pd.DataFrame({
    "id": df["id"], "season": df["season"], "startDate": df["startDate"],
    "team": df["homeTeam"], "won": (df["homePoints"] > df["awayPoints"]).astype(int),
})
away_side = pd.DataFrame({
    "id": df["id"], "season": df["season"], "startDate": df["startDate"],
    "team": df["awayTeam"], "won": (df["awayPoints"] > df["homePoints"]).astype(int),
})
long = (pd.concat([home_side, away_side])
          .sort_values(["team", "season", "startDate"])
          .reset_index(drop=True))

def streak_before(wins: pd.Series) -> pd.Series:
    """Positive = consecutive wins coming into the game, negative = consecutive losses."""
    out, cur = [], 0
    for w in wins:
        out.append(cur)                      # value BEFORE this game
        if w == 1:
            cur = cur + 1 if cur > 0 else 1
        else:
            cur = cur - 1 if cur < 0 else -1
    return pd.Series(out, index=wins.index)

g = long.groupby(["team", "season"])["won"]
long["streak"] = g.transform(streak_before)
long["games_played"] = g.cumcount()
long["wins_before"] = g.transform(lambda s: s.shift(1).fillna(0).cumsum())
long["win_pct_before"] = np.where(long["games_played"] > 0,
                                  long["wins_before"] / long["games_played"], np.nan)
long["won_last_game"] = g.shift(1)

form_cols = ["streak", "win_pct_before", "won_last_game", "games_played"]
form = long[["id", "team"] + form_cols]

df = df.merge(form.rename(columns={"team": "homeTeam", **{c: f"home_{c}" for c in form_cols}}),
              on=["id", "homeTeam"], how="left")
df = df.merge(form.rename(columns={"team": "awayTeam", **{c: f"away_{c}" for c in form_cols}}),
              on=["id", "awayTeam"], how="left")

# sanity check: a team's first game of the season must start with a streak of 0
assert (df.loc[df["home_games_played"] == 0, "home_streak"] == 0).all()
df[["season", "week", "homeTeam", "home_streak", "home_win_pct_before", "home_won_last_game"]].head(10)

,season,week,homeTeam,home_streak,home_win_pct_before,home_won_last_game
0,2024,1,Georgia Tech,0,NaN,NaN
1,2024,1,New Mexico,0,NaN,NaN
2,2024,1,Nevada,0,NaN,NaN
3,2024,1,Hawai'i,0,NaN,NaN
4,2024,1,Rutgers,0,NaN,NaN
5,2024,1,Buffalo,0,NaN,NaN
6,2024,1,Central Michigan,0,NaN,NaN
7,2024,1,Bowling Green,0,NaN,NaN
8,2024,1,NC State,0,NaN,NaN
9,2024,1,UCF,0,NaN,NaN


**Caveat:** the dataset only covers games hosted by FBS teams, so a few road games
(e.g. an FBS team playing at an FCS school) are missing and streaks can be slightly off.

## 4. Kickoff timing (in the stadium's local time)

In [5]:
local = [ts.tz_convert(tz) if isinstance(tz, str) else pd.NaT
         for ts, tz in zip(df["startDate"], df["timezone"])]
local_hour = pd.Series([t.hour + t.minute / 60 if t is not pd.NaT else np.nan for t in local])
local_dow = pd.Series([t.dayofweek if t is not pd.NaT else np.nan for t in local])

# TBD start times are stored as midnight UTC, so the hour is meaningless for those
df["kickoff_hour"] = np.where(df["startTimeTBD"], np.nan, local_hour)
df["day_of_week"] = local_dow            # 0 = Monday ... 5 = Saturday
df["is_saturday"] = (df["day_of_week"] == 5).astype(int)
df["night_game"] = (df["kickoff_hour"] >= 18).astype(int)
df["early_season"] = (df["week"] <= 3).astype(int)
df["late_season"] = (df["week"] >= 12).astype(int)
df[["kickoff_hour", "day_of_week"]].describe().T

,count,mean,std,min,25%,50%,75%,max
kickoff_hour,2020.0,16.025000,2.877452,10.0,14.0,16.0,18.75,20.5
day_of_week,2021.0,4.725383,0.784310,0.0,5.0,5.0,5.00,6.0


## 5. Venue and travel

In [6]:
df["stadium_age"] = df["season"] - df["constructionYear"]

# Approximate each team's home location from its own non-neutral home games
home_loc = (df[~df["neutralSite"]]
            .groupby("homeTeam")[["latitude", "longitude"]].median())

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

away_lat = df["awayTeam"].map(home_loc["latitude"])
away_lon = df["awayTeam"].map(home_loc["longitude"])
df["away_travel_km"] = haversine_km(df["latitude"], df["longitude"], away_lat, away_lon)
df["away_travel_missing"] = df["away_travel_km"].isna().astype(int)   # mostly FCS opponents
df["same_state_opponent"] = (
    df["awayTeam"].map(df.drop_duplicates("homeTeam").set_index("homeTeam")["state"]) == df["state"]
).astype(int)

for c in ["grass", "dome", "neutralSite", "conferenceGame"]:
    df[c] = df[c].astype(int)

print(df["away_travel_missing"].mean().round(3), "of games have no travel distance")
df[["stadium_age", "away_travel_km", "same_state_opponent"]].describe().T

0.171 of games have no travel distance


,count,mean,std,min,25%,50%,75%,max
stadium_age,2021.0,62.899060,32.376723,2.00000,35.000000,61.000000,95.000000,113.000000
away_travel_km,1676.0,1094.150300,1015.313187,4.14242,388.478251,779.272945,1428.944504,8463.606099
same_state_opponent,2021.0,0.106878,0.309034,0.00000,0.000000,0.000000,0.000000,1.000000


## 6. Assemble the modeling table

The target is `high_attendance` (attendance at 90%+ of capacity). Rows without a label
(neutral sites, missing attendance or capacity) are dropped.

In [7]:
LEAKAGE = [
    "attendance", "attendance_pct", "flag_over_capacity",          # derived from the target
    "homePoints", "awayPoints", "homeLineScores", "awayLineScores",
    "homePostgameElo", "awayPostgameElo",
    "homePostgameWinProbability", "awayPostgameWinProbability",
    "excitementIndex",
]

FEATURES = [
    # existing pregame columns
    "homePregameElo", "awayPregameElo", "capacity", "elevation", "grass", "dome",
    "conferenceGame", "temperature_c", "precip_mm", "windspeed_kmh", "week",
    # engineered
    "elo_diff", "abs_elo_diff", "combined_elo", "away_is_fcs", "away_is_power", "home_is_power",
    "rivalry_game",
    "home_streak", "home_win_pct_before", "home_won_last_game",
    "away_streak", "away_win_pct_before", "away_won_last_game",
    "kickoff_hour", "day_of_week", "is_saturday", "night_game", "early_season", "late_season",
    "stadium_age", "away_travel_km", "away_travel_missing", "same_state_opponent",
]
CATEGORICAL = ["homeConference"]
ID_COLS = ["id", "season", "homeTeam", "awayTeam"]
TARGET = "high_attendance"

assert not set(FEATURES) & set(LEAKAGE), "a leakage column slipped into FEATURES"

model_df = df.loc[df[TARGET].notna() & ~df["neutralSite"].astype(bool),
                  ID_COLS + FEATURES + CATEGORICAL + [TARGET]].copy()
model_df[TARGET] = model_df[TARGET].astype(int)
print(model_df.shape)
print(model_df[TARGET].value_counts(normalize=True).round(3))
model_df.isna().sum()[lambda s: s > 0]

(1673, 40)
high_attendance
0    0.556
1    0.444
Name: proportion, dtype: float64


home_win_pct_before    172
home_won_last_game     172
away_win_pct_before    257
away_won_last_game     257
kickoff_hour             1
away_travel_km         232
dtype: int64

## 7. Quick check: how do the new features relate to the target?

In [8]:
new_feats = ["elo_diff", "abs_elo_diff", "combined_elo", "away_is_fcs", "away_is_power",
             "rivalry_game", "home_streak", "home_win_pct_before", "night_game",
             "stadium_age", "away_travel_km", "same_state_opponent"]
model_df[new_feats + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3)

combined_elo           0.497
away_is_power          0.380
home_win_pct_before    0.291
stadium_age            0.252
elo_diff               0.231
home_streak            0.209
rivalry_game           0.166
abs_elo_diff           0.107
same_state_opponent    0.086
away_is_fcs           -0.056
night_game             0.015
away_travel_km         0.002
Name: high_attendance, dtype: float64

In [9]:
print("High-attendance rate, rivalry vs not:")
print(model_df.groupby("rivalry_game")[TARGET].agg(["mean", "count"]).round(3))

High-attendance rate, rivalry vs not:
               mean  count
rivalry_game              
0             0.424   1583
1             0.789     90


## 8. Save
`homeConference` is left as text; one-hot encode it inside the modeling pipeline.
Split by **season** (e.g. train 2024–2025, test 2026) rather than randomly, since streak and
win-% features are built from earlier games in the same season.

In [10]:
model_df.to_csv("../data/features.csv", index=False)
print("saved", model_df.shape)

saved (1673, 40)
